# 1. Ingesta y depuración base

Este notebook funciona como **inicio del proceso metodológico de construcción de la base canónica**. Su objetivo es documentar, de manera reproducible, cómo se parte de los registros operativos crudos y se arriba a una versión consistente de `reclamos`, `tareas` y referencias geográficas mínimas para el resto del trabajo.

En esta etapa se fija la fuente temporal canónica de `tramites`, se aplica la ventana analítica `2021-01-01` → `2026-03-31`, se normalizan y corrigen los campos geográficos `geo1` y `geo2`, se auditan exclusiones y se persisten los outputs procesados dentro de `ceia-amq-tp/`.

## Cómo leer este notebook

1. **Inventario de entradas**: verifica qué fuentes participan y desde dónde se leen.
2. **Resumen de corrida**: muestra cuánto volumen se conserva, cuánto se excluye y qué transformaciones relevantes ocurrieron.
3. **Artifacts resultantes**: deja trazabilidad explícita de los datasets persistidos para etapas posteriores.
4. **Muestras de salida**: permite inspeccionar la forma final de reclamos, tareas y destinos únicos.

> Alcance: depuración, estandarización y auditabilidad de la capa operativa base.  
> Fuera de alcance: análisis exploratorio, modelado y conclusiones sustantivas del trabajo final.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from helpers.paths import MANDATORY_INPUTS, require_input
from helpers.reclamos_cleaning import DATE_WINDOW_END, DATE_WINDOW_START, run_phase2_pipeline


## 1. Entradas requeridas y criterio de consistencia

Antes de ejecutar la limpieza conviene fijar **qué fuentes intervienen**. La tabla siguiente no solo verifica disponibilidad: también deja claro que esta notebook integra reclamos, tareas y referencias auxiliares bajo una única convención temporal y geográfica. Esa trazabilidad es importante porque cualquier inconsistencia en esta etapa se propaga al resto del pipeline.

Como nota metodológica, el input `lluvias` que aparece en ese inventario **no es un registro primario capturado directamente por esta notebook**, sino un artefacto derivado generado upstream a partir de planillas pluviométricas escaneadas mediante OCR/extracción estructurada con Google GenAI API (Gemini). Acá no se reprocesan esos documentos: este notebook solo consume el resultado tabular ya consolidado como dependencia para mantener la alineación temporal del pipeline más amplio.


In [2]:
required = {name: require_input(name) for name in MANDATORY_INPUTS}
pd.DataFrame({
    'dataset': list(required.keys()),
    'path': [str(path.relative_to(ROOT)) for path in required.values()]
})

,dataset,path
0,tramites,data/raw/ceml/tramites.parquet
1,tramites_tareas,data/raw/ceml/tramites_tareas.parquet
2,costos_vehiculos,data/raw/archivos/costos_vehiculos.json
3,sedes_servicios,data/raw/archivos/sedes_servicios.json
4,tarifas_ceml,data/raw/archivos/tarifas_ceml_2026_03.json
5,tasas_pdf,data/raw/archivos/Tasas 03-2026.pdf
6,lluvias,data/raw/rain/dataset_lluvias_diario_con_obs.csv


## 2. Resultado de la depuración

La siguiente salida resume el efecto de las reglas aplicadas. Acá importa leer tres cosas: **volumen retenido**, **exclusiones justificadas** y **ajustes correctivos** como desajustes temporales o intercambios detectados en campos geográficos. No es solo una corrida técnica: es la evidencia de por qué el dataset final puede considerarse defendible.


In [3]:
result = run_phase2_pipeline()
summary = pd.Series(result['summary'], name='value').rename_axis('metric').reset_index()
summary

,metric,value
0,tramites_total,372551
1,tramites_clean,159030
2,tramites_excluded,213521
3,tramites_temporal_mismatch,16
4,tramites_geo_swapped,718
5,tareas_total,579
6,tareas_clean,254
7,tareas_excluded,325
8,destinos_unicos,14774


## 3. Artifacts persistidos para las siguientes etapas

Una vez cerrada la limpieza, el notebook deja datasets intermedios y finales listos para reutilización. La tabla siguiente sirve como mapa de continuidad entre notebooks: muestra **qué se guardó** y **dónde vive** cada output.


In [4]:
pd.DataFrame({
    'artifact': list(result['artifacts'].keys()),
    'path': [str(Path(path).relative_to(ROOT)) for path in result['artifacts'].values()]
})

,artifact,path
0,reclamos_clean,data/processed/reclamos_clean.parquet
1,tareas_clean,data/processed/tareas_clean.parquet
2,destinos_unicos,data/processed/destinos_unicos.parquet
3,exclusiones_reclamos,data/intermediate/exclusiones_reclamos.parquet
4,exclusiones_tareas,data/intermediate/exclusiones_tareas.parquet
5,tramites_temporal_audit,data/intermediate/tramites_temporal_audit.parquet
6,report,reports/phase2_cleaning_audit.md


## 4. Inspección rápida de la estructura final

Las vistas siguientes no buscan agotar el análisis, sino comprobar que la salida ya tiene forma operativa útil. Primero se observa la estructura de `reclamos_clean`, luego la de `tareas_clean` y finalmente la tabla de `destinos_unicos` que consolida la base geográfica mínima necesaria para costeo y ruteo.


In [5]:
reclamos_clean = pd.read_parquet(result['artifacts']['reclamos_clean'])
tareas_clean = pd.read_parquet(result['artifacts']['tareas_clean'])
destinos_unicos = pd.read_parquet(result['artifacts']['destinos_unicos'])
{
    'temporal_window_start': DATE_WINDOW_START,
    'temporal_window_end': DATE_WINDOW_END,
    'reclamos_clean_shape': reclamos_clean.shape,
    'tareas_clean_shape': tareas_clean.shape,
    'destinos_unicos_shape': destinos_unicos.shape,
}

{'temporal_window_start': Timestamp('2021-01-01 00:00:00'),
 'temporal_window_end': Timestamp('2026-03-31 23:59:59'),
 'reclamos_clean_shape': (159030, 15),
 'tareas_clean_shape': (254, 7),
 'destinos_unicos_shape': (14774, 4)}

In [6]:
reclamos_clean[['reclamo_id', 'fecha_reclamo', 'lat', 'lon', 'estado_geo', 'motivo', 'servicio']].head(10)

,reclamo_id,fecha_reclamo,lat,lon,estado_geo,motivo,servicio
0,147803-38982,2021-01-01 05:14:00,-26.560068,-54.784101,direct,SIN ENERGIA,Energia
1,147805-38984,2021-01-01 16:46:00,-26.551452,-54.719487,direct,SIN ENERGIA,Energia
2,147806-38985,2021-01-01 18:55:00,-26.567776,-54.730573,direct,SIN ENERGIA,Energia
3,147807-38986,2021-01-01 20:26:00,-26.568716,-54.749879,direct,BAJA TENSION,Energia
4,147808-38987,2021-01-01 20:26:00,-26.472822,-54.714013,direct,SIN ENERGIA,Energia
5,147810-7298,2021-01-02 07:41:00,-26.537925,-54.649263,direct,FALTA DE AGUA,Agua Potable
6,147811-38989,2021-01-02 07:55:00,-26.572488,-54.733239,direct,SIN ENERGIA,Energia
7,147812-38990,2021-01-02 09:00:00,-26.595006,-54.734545,direct,VARIOS,Energia
8,147814-38992,2021-01-02 11:35:00,-26.510873,-54.591587,direct,SIN ENERGIA,Energia
9,147815-12295,2021-01-02 12:30:00,-26.726105,-54.721433,direct,SIN SEÑAL,TV Cable


In [7]:
tareas_clean[['reclamo_id', 'tarea_id', 'fecha_tarea', 'tipo_tarea']].head(10)

,reclamo_id,tarea_id,fecha_tarea,tipo_tarea
0,150634-4753,1,2021-02-05 09:34:00,Se debe realizar ampliacion de red para traslado.
1,148559-7337,1,2021-02-05 16:00:00,Falta reparar - Verificado Por Avalos.
2,151688-2382,1,2021-03-03 13:00:00,"Verificado, tiene todo para la conexion, se de..."
3,149753-2345,1,2021-03-03 15:00:00,"Verificado, No Cuenta con el Pilar Reglamentar..."
4,154160-7678,1,2021-03-29 06:10:00,donde se solicito la conexion no hay red de ag...
5,155342-7735,1,2021-04-13 11:30:00,Hay qye realizar extension de la Red. A la esp...
6,154560-2411,1,2021-04-21 13:00:00,"Verificado, tiene instalado solo el pilar."
7,155593-2429,1,2021-05-12 06:00:00,Se debe realizar extension de red.
8,157386-2450,1,2021-05-19 13:00:00,"Al momento de la fecha no cuenta con pilar, ni..."
9,153474-2403,1,2021-06-04 10:00:00,"DEBE RUBICAR EL PILAR DE AGUA, QUEDA EN AVISAR..."


In [8]:
destinos_unicos.head(10)

,destino_key,lat,lon,reclamos_count
0,-26.56581_-54.77084,-26.56581,-54.77084,877
1,-26.56847_-54.75395,-26.56847,-54.75395,449
2,-26.56839_-54.75431,-26.56839,-54.75431,309
3,-26.56838_-54.75435,-26.56838,-54.75435,294
4,-26.56874_-54.75527,-26.56874,-54.75527,288
5,-26.57037_-54.57839,-26.57037,-54.57839,234
6,-26.55822_-54.75270,-26.55822,-54.75270,208
7,-26.56625_-54.75473,-26.56625,-54.75473,172
8,-26.56830_-54.75434,-26.56830,-54.75434,162
9,-26.56834_-54.75433,-26.56834,-54.75433,158


## Decisiones auditadas

- Fuente temporal canónica de `tramites`: `dt_inicio`.
- Regla temporal explícita y alineada con lluvias: conservar solo reclamos entre `2021-01-01` y `2026-03-31` inclusive.
- Mapeo reclamo ↔ tarea: `reclamo_id = NumeroTramite-NumeroOrden`, `codigotar -> tarea_id`, `fecha/hora -> fecha_tarea`, `tarea -> tipo_tarea`.
- Geolocalización: `geo1/geo2` se coercionan a numérico, se corrigen filas invertidas y se excluye lo faltante, cero o fuera de bounds.
- Auditoría detallada: ver `reports/phase2_cleaning_audit.md` y `data/intermediate/tramites_temporal_audit.parquet`.